[Guide Index](README.md)

# 04a SRL: how a rule set runs

[04](04-srl-sparql-rules.ipynb) shows how to write rules. This guide explains how a rule set runs: the order its rules run in (the plan), the rules that run once, the rule sets that are rejected and why, what running the rules again adds, and `DATA` blocks and imports. The examples use a software-dependency graph.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later sections may reuse variables from earlier ones.

In [1]:
from starlayer.graph import NamedNode, StarLayerGraph
from starlayer.sparql import srl

EX = "http://example.org/"


def ex(name):
    """A term in the example namespace, as a pyoxigraph NamedNode."""
    return NamedNode(EX + name)


def show(graph):
    """Print a graph as pretty Turtle, declaring only the prefixes it uses.
    (A new StarLayerGraph has no prefixes; the ones used here are bound explicitly.)"""
    print(graph.serialize(format="turtle", pretty=True, only_used=True))

## 1. The base graph

The examples use a small software-dependency graph, the same running example as [the SRL spec's worked example](https://www.w3.org/TR/sparql12-rl/#worked-example): a `:frontend` depends on an `:app`, which depends on a `:db` that has a known vulnerability. A `:logger` component has no vulnerabilities at all. The rules below spread the vulnerability along the dependency chain.

In [2]:
base = StarLayerGraph(prefixes={"ex": EX})
base.parse(data="""
    @prefix ex: <http://example.org/> .

    ex:frontend a ex:Component ; ex:dependsOn ex:app .
    ex:app      a ex:Component ; ex:dependsOn ex:db .
    ex:db       a ex:Component ; ex:hasVulnerability ex:vuln1 .
    ex:logger   a ex:Component .  # no vulnerability - stays safe in section 4

    ex:vuln1 ex:severity 9.1 .
""", format="turtle")


print("data graph")
show(base)

data graph
@prefix ex: <http://example.org/> .

ex:app a ex:Component ;
    ex:dependsOn ex:db .

ex:db a ex:Component ;
    ex:hasVulnerability ex:vuln1 .

ex:frontend a ex:Component ;
    ex:dependsOn ex:app .

ex:logger a ex:Component .

ex:vuln1 ex:severity 9.1 .



## 2. The rules

Four rules work on this graph. R1 and R2 find which components are exposed to a vulnerability: R1 for a vulnerability a component has itself, and R2, which is recursive, for one it reaches through `ex:dependsOn`. R3 marks a component critically exposed if a vulnerability has `severity >= 9.0` (`FILTER`). R4 marks a component safe if it is *not* critically exposed (`NOT { }`).

R4 needs the result of R3, and R3 needs the result of R1 and R2. The next section shows how SRL works out an order for them.

In [3]:
rules_text = """
PREFIX ex: <http://example.org/>

# R1: a component is exposed to any vulnerability it has directly.
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }

# R2: exposure propagates along dependsOn (recursive - depends on itself).
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

# R3: severity >= 9.0 counts as a critical vulnerability.
RULE { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }

# R4: not critically exposed -> safe to deploy.
RULE { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }
"""

ruleset = srl.parse_ruleset(rules_text)
inferred = ruleset.infer(base)
inferred.bind("ex", EX)
show(inferred)

@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:logger ex:status ex:safeToDeploy .



## 3. The plan

Before running anything, SRL works out an order for the rules. `ruleset.plan()` shows it: the **strata** (stages) in the order they run, and the **dependency edges** between rules that the strata were worked out from. Rules are identified by their position in `ruleset.rules`, so R1 is index 0.

Each stratum finishes before the next starts. Within a stratum the ordinary rules repeat until nothing new appears.

An edge `source -> target` means the `source` rule's body reads something the `target` rule's head can produce. It is **open** for an ordinary dependency: the two can sit in the same stratum and repeat together. It is **closed** when the source must see everything the target produces before it runs: when it reads the target through `NOT { }`, or when the source is a run-once rule (next section). A closed edge forces the source into a *later* stratum than the target.

For the four rules above, R4 has a closed edge to R3, so it waits for the stratum after R3:

In [4]:
plan = ruleset.plan()

for stratum in plan.strata:
    print(f"stratum {stratum.number}:", ", ".join(f"R{r.index + 1}" for r in stratum.general))

for edge in sorted(plan.edges, key=lambda e: (e.source, e.target)):
    print(f"R{edge.source + 1} -> R{edge.target + 1} ({edge.label})")

assert [[r.index for r in s.general] for s in plan.strata] == [[0, 1, 2], [3]]
assert {(e.source, e.target): e.label for e in plan.edges}[(3, 2)] == "closed"

stratum 0: R1, R2, R3
stratum 1: R4
R2 -> R1 (open)
R2 -> R2 (open)
R3 -> R1 (open)
R3 -> R2 (open)
R4 -> R3 (closed)


### Rule order doesn't matter

The plan comes from the rules' *structure*, not from the order they are written in. Below, R4 is written and listed before R3, and each rule is given an id so the plan can name it. R4 still lands in the later stratum, and `infer()` produces exactly the same result.

In [5]:
reordered_rules_text = """
PREFIX ex: <http://example.org/>

RULE ex:R1 { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v . }
RULE ex:R2 { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v . }

# R4 listed before R3 on purpose - it has a closed dependency on R3 through NOT { }.
RULE ex:R4 { ?x ex:status ex:safeToDeploy }
WHERE { ?x a ex:Component . NOT { ?x ex:status ex:criticallyExposed } }

RULE ex:R3 { ?x ex:status ex:criticallyExposed }
WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . FILTER(?s >= 9.0) }
"""

reordered_ruleset = srl.parse_ruleset(reordered_rules_text)

for stratum in reordered_ruleset.plan().strata:
    print(f"stratum {stratum.number}:", ", ".join(base.qname(r.id) for r in stratum.general))

reordered_inferred = reordered_ruleset.infer(base)
print("same result regardless of rule order:", set(reordered_inferred) == set(inferred))
assert set(reordered_inferred) == set(inferred)

stratum 0: ex:R1, ex:R2, ex:R3
stratum 1: ex:R4
same result regardless of rule order: True


## 4. Run-once rules

Some rules run exactly once, at their place in the plan, instead of repeating. A rule is a **run-once rule** when:
- its body has a `SET` or a group element, or
- its head has a blank node, or a triple term that contains a variable.

These are the rules that can create terms that were not in the graph before. A rule that only reuses terms already in the graph can only ever produce a finite number of triples, so the repeated passes must stop. A rule that makes new values cannot be repeated freely: it could go on creating new ones for ever. So it runs once instead.

A run-once rule waits for every rule it reads from to finish: all its dependencies are closed. The rules that read its result come after it. Within a stratum the run-once rules run first, one time each, and then the ordinary rules repeat.

Below, `score` is a run-once rule: it computes a risk score from the exposure rules, so it runs after them. `high` is an ordinary rule that reads the score, and `score2` is a second run-once rule that reads `score`:

In [6]:
staged_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE ex:exposure1 { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v }
RULE ex:exposure2 { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v }
RULE ex:score     { ?x ex:score ?r }     WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . SET(?r := ?s * 10) }
RULE ex:high      { ?x ex:high true }    WHERE { ?x ex:score ?r . FILTER(?r > 50) }
RULE ex:score2    { ?x ex:score2 ?r }    WHERE { ?x ex:score ?s . SET(?r := ?s + 1) }
RULE ex:unrelated { ?x ex:q ?y }         WHERE { ?x ex:p ?y }
""")

for stratum in staged_rules.plan().strata:
    once = ", ".join(base.qname(r.id) for r in stratum.run_once) or "-"
    repeat = ", ".join(base.qname(r.id) for r in stratum.general) or "-"
    print(f"stratum {stratum.number}: run once: {once:12} | repeat: {repeat}")

stratum 0: run once: -            | repeat: ex:exposure1, ex:exposure2, ex:unrelated
stratum 1: run once: ex:score     | repeat: ex:high
stratum 2: run once: ex:score2    | repeat: -


`score` and `high` share a stratum: `score` runs once first, and then `high` repeats. `score2` is a run-once rule that reads what `score` makes, so it has a stratum of its own after `score`. Run-once rules can depend on each other. What they cannot do is depend on each other in a loop, as the next section shows.

## 5. Cycles and rejected rule sets

A rule **depends on** another when a pattern in its body could match a triple the other rule's head creates. The check compares the rules' patterns, not the data. A **cycle** is a loop of such dependencies, including a rule that depends on itself.

Cycles are normal for ordinary rules. R2 depends on itself, and two ordinary rules can depend on each other: they repeat together until nothing new appears. What is not allowed is a cycle that goes through a **closed** dependency, which is `NOT { }`, a group element or a run-once rule. That rule would have to wait for the rules it reads from to finish, and one of them is itself. It would be waiting for itself, and no order exists.

For these rule sets the specification leaves the outcome undefined, and StarLayer raises `StratificationError`, from `plan()` and from `infer()`. The check is on the patterns, so a rule that *could* match its own output is rejected even if it would stop in practice. The cases below show what is accepted and what is not:

In [7]:
from starlayer.sparql.srl_eval import StratificationError

PREFIXES = "PREFIX ex: <http://example.org/>\n"

cases = [
    ("a rule that depends on itself",
     "RULE { ?x ex:r ?z } WHERE { ?x ex:p ?y . ?y ex:r ?z } RULE { ?x ex:r ?y } WHERE { ?x ex:p ?y }"),
    ("two ordinary rules that depend on each other",
     "RULE { ?x ex:even ?y } WHERE { ?x ex:p ?z . ?z ex:odd ?y } RULE { ?x ex:odd ?y } WHERE { ?x ex:p ?y } "
     "RULE { ?x ex:odd ?y } WHERE { ?x ex:p ?z . ?z ex:even ?y }"),
    ("a chain of run-once rules (no cycle)",
     "RULE { ?x ex:b ?n } WHERE { ?x ex:a ?m . SET(?n := ?m + 1) } RULE { ?x ex:c ?n } WHERE { ?x ex:b ?m . SET(?n := ?m + 1) }"),
    ("a rule that negates its own output (NOT)",
     "RULE { ?x ex:flagged ex:yes } WHERE { ?x a ex:Component . NOT { ?x ex:flagged ex:yes } }"),
    ("a SET rule that feeds itself",
     "RULE { ?x ex:level ?n } WHERE { ?x ex:level ?m . SET(?n := ?m + 1) }"),
    ("two run-once rules that feed each other",
     "RULE { ?x ex:b ?n } WHERE { ?x ex:a ?m . SET(?n := ?m + 1) } RULE { ?x ex:a ?n } WHERE { ?x ex:b ?m . SET(?n := ?m + 1) }"),
    ("a blank node feeding itself",
     "RULE { ?x ex:child _:c } WHERE { ?x ex:child ?y }"),
]

for label, text in cases:
    try:
        srl.parse_ruleset(PREFIXES + text).plan()
        print(f"{label:48} accepted")
    except StratificationError:
        print(f"{label:48} StratificationError")

a rule that depends on itself                    accepted
two ordinary rules that depend on each other     accepted
a chain of run-once rules (no cycle)             accepted
a rule that negates its own output (NOT)         StratificationError
a SET rule that feeds itself                     StratificationError
two run-once rules that feed each other          StratificationError
a blank node feeding itself                      StratificationError


## 6. Validation errors

A rule set can be wrong before it runs. StarLayer reports it when the rule set is parsed:

| What is wrong | Error | When |
|---|---|---|
| a variable in a rule's head is not bound by its body, or the text is not valid SRL, or it uses a property path other than `/` and `^` | `SRLParseError` | `srl.parse_ruleset()` |
| the rule set does not conform to StarLayer's SHACL shapes for SRL | `SRLParseError` | `srl.parse_ruleset()` |
| a cycle goes through a closed dependency (section 5) | `StratificationError` | `plan()` and `infer()` |

Every variable used in a rule's head must be bound by its body ([SRL section 4.2](https://www.w3.org/TR/sparql12-rl/#well-formedness-conditions)). A rule set that breaks this raises `SRLParseError` when it is parsed, so a typo in a variable name does not give a wrong answer later. The check against the SHACL shapes takes about a tenth of a second, so a program that parses many rule sets it already trusts can pass `validate=False` to skip it.

In [8]:
try:
    broken = srl.parse_ruleset("""
    PREFIX ex: <http://example.org/>
    RULE { ?x ex:flaggedBy ?reviewer } WHERE { ?x ex:hasVulnerability ?v . }
    """)
except srl.SRLParseError as e:
    print("SRLParseError:", e)


SRLParseError: SRL: rule set is not well-formed per §4.2: unbound_head_variable (?reviewer in rule head)


## 7. Running a rule set again

A rule set derives everything it can in one call to `infer()`. If you run the same rules over the base graph together with what they inferred, ordinary rules add nothing new, because the result is already complete. Rules with a `SET` or a group element add nothing either, since they depend only on rules that have finished.

Two kinds of rule can add triples on a second run:
- a rule with a **blank node** in its head, which creates fresh nodes on every run;
- a rule that uses **`WHERE DATA`** (section 8), which sees the earlier results as base data on the second run.

Whether to run again, and how many times, is up to the application. The function below counts the triples a second run adds for each kind of rule:

In [9]:
EXPOSURE = """
PREFIX ex: <http://example.org/>
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v }
RULE { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v }
"""


def second_run(rules_text):
    """The number of triples a rule set derives, and how many a second run over the base graph plus those adds."""
    rules = srl.parse_ruleset(rules_text)
    first = rules.infer(base)
    combined = base.copy()
    combined += first
    again = rules.infer(combined)
    return len(first), sum(1 for triple in again.triples() if triple not in combined)


examples = {
    "ordinary rules": EXPOSURE,
    "a SET rule": EXPOSURE + "RULE { ?x ex:score ?r } WHERE { ?x ex:exposedTo ?v . ?v ex:severity ?s . SET(?r := ?s * 10) }",
    "a blank node in a head": EXPOSURE + "RULE { ?x ex:ticket _:t } WHERE { ?x ex:exposedTo ?v }",
    "WHERE DATA": "PREFIX ex: <http://example.org/> "
                  "RULE { ?x ex:flagged ex:yes } WHERE DATA { ?x ex:hasVulnerability ?v } "
                  "RULE { ?y ex:seenAsFlagged ex:yes } WHERE DATA { ?y ex:flagged ex:yes }",
}

for label, text in examples.items():
    derived, added = second_run(text)
    print(f"{label:24} first run: {derived} triples | a second run adds: {added}")

ordinary rules           first run: 3 triples | a second run adds: 0


a SET rule               first run: 6 triples | a second run adds: 0
a blank node in a head   first run: 6 triples | a second run adds: 3


WHERE DATA               first run: 1 triples | a second run adds: 1


## 8. `DATA` blocks

A rule set can carry its own ground facts in a `DATA { }` block.  The facts travel with the rules and get combined with the base graph before the rules are run. 

A rule matches them exactly as it matches triples from the base graph, and they also appear in the output.

The spec names three graphs, and this section depends on telling them apart:
- the **base graph** is the data you pass to `infer()`;
- the **evaluation graph** is what the rules match as they run: the base graph, plus the rule set's `DATA { }` facts, plus every triple inferred so far;
- the **inference graph** is what `infer()` returns: the inferred triples, not including anything already in the base graph.

A `DATA { }` block behaves like a rule with an empty body: its triples join the evaluation graph without any rule being run.

Here the rule set records which team owns two of the components, and one rule derives a `reportsTo` triple from each:

In [10]:
# `base` is the software-dependency graph from section 1
team_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . ex:app ex:team ex:web . }

RULE { ?x ex:reportsTo ?t } WHERE { ?x ex:team ?t }
""")

team_inferred = team_rules.infer(base)
team_inferred.bind("ex", EX)
show(team_inferred)

assert (ex("db"), ex("team"), ex("platform")) in team_inferred      # the DATA facts are in the output
assert (ex("db"), ex("reportsTo"), ex("platform")) in team_inferred # and the rule matched them

@prefix ex: <http://example.org/> .

ex:app ex:reportsTo ex:web ;
    ex:team ex:web .

ex:db ex:reportsTo ex:platform ;
    ex:team ex:platform .



### `WHERE DATA` and `NOT DATA`: matching base data

A rule normally matches the whole evaluation graph. Writing `DATA` after `WHERE` (or after a `NOT`) changes that to **base data**: the rule then matches only the base graph.

The keyword is easy to misread. `WHERE DATA` does **not** mean "match the `DATA { }` block". The base graph is the data you passed to `infer()`, and it is separate from the rule set's own `DATA { }` facts.

| A rule's `WHERE` matches... | plain `WHERE { }` | `WHERE DATA { }` |
|---|---|---|
| the base graph passed to `infer()` | yes | yes |
| the rule set's own `DATA { }` facts | yes | **no** |
| triples inferred by the rules so far | yes | **no** |

Use it for a rule that should look only at the data you supplied, never at the rule set's own facts or at other rules' conclusions. The spec's section "Matching Base Data" (§3.9) says the same: matching on the base graph "does not include triples added by rule set evaluation from the rule set DATA blocks".

The same `DATA` fact is used below by three rules:
- the first uses a plain `WHERE`, so it matches the `DATA` fact;
- the second uses `WHERE DATA` for the same pattern, so it matches nothing, because `ex:team` exists only in the `DATA` block, not in the base graph;
- the third uses `WHERE DATA` for `ex:hasVulnerability`, which *is* in the base graph, so it matches.

In [11]:
# `base` is the software-dependency graph from section 1
scope_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>

DATA { ex:db ex:team ex:platform . }

RULE { ?x ex:seenInData ?t }  WHERE      { ?x ex:team ?t }
RULE { ?x ex:seenInBase ?t }  WHERE DATA { ?x ex:team ?t }
RULE { ?x ex:flagged ex:yes } WHERE DATA { ?x ex:hasVulnerability ?v }
""")

scope_inferred = scope_rules.infer(base)
scope_inferred.bind("ex", EX)
show(scope_inferred)

assert (ex("db"), ex("seenInData"), ex("platform")) in scope_inferred       # ordinary WHERE sees the DATA fact
assert not list(scope_inferred.triples((None, ex("seenInBase"), None)))  # WHERE DATA does not
assert (ex("db"), ex("flagged"), ex("yes")) in scope_inferred               # WHERE DATA sees the base graph

@prefix ex: <http://example.org/> .

ex:db ex:flagged ex:yes ;
    ex:seenInData ex:platform ;
    ex:team ex:platform .


### `NOT DATA`: filling in a default

`DATA` can also be used on a single `NOT`. The spec's own use is a **default value**: derive a value only when the base graph does not already state one. The rule below gives every vulnerable component a review priority unless the base graph already has one. The cell builds its own small base graph (just `ex:db`, with a vulnerability) and runs the rule three ways: with no priority in the base graph, with one already there, and with a plain `NOT` instead of `NOT DATA`.

The rule negates the very predicate it produces (`ex:reviewPriority`). With a plain `NOT { }` the rule cannot be ordered (section 5), because the rule would need its own output to be finished before it could run. `NOT DATA { }` looks only at the base graph, so there is no cycle, and the rest of the rule body can still match inferred triples.

In [12]:
# The base graph for this example: ex:db is a component with a known vulnerability.
# (Section 1's graph is not used here, so the example stands on its own.)
review_base = StarLayerGraph(prefixes={"ex": EX})
review_base.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:db a ex:Component ; ex:hasVulnerability ex:vuln1 .
""", format="turtle")

# The rule: a component with a vulnerability gets a high review priority,
# unless the base graph already states a priority for it.
default_rules = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:reviewPriority ex:high }
WHERE { ?x ex:hasVulnerability ?v . NOT DATA { ?x ex:reviewPriority ?existing } }
""")

# Case 1: the base graph has no priority for ex:db, so the default is derived.
case1 = default_rules.infer(review_base)
case1.bind("ex", EX)
print("Case 1 - base graph has no priority. Derived:")
show(case1)

# Case 2: the base graph already states a priority for ex:db, so nothing is derived.
review_base_with_priority = StarLayerGraph()
review_base_with_priority += review_base
review_base_with_priority.add((ex("db"), ex("reviewPriority"), ex("low")))
case2 = default_rules.infer(review_base_with_priority)
print("Case 2 - base graph already has ex:db ex:reviewPriority ex:low. Triples derived:", len(case2))

assert (ex("db"), ex("reviewPriority"), ex("high")) in case1
assert len(case2) == 0

# Case 3: the same rule with a plain NOT (matching the whole evaluation graph) cannot be ordered.
plain_not = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
RULE { ?x ex:reviewPriority ex:high }
WHERE { ?x ex:hasVulnerability ?v . NOT { ?x ex:reviewPriority ?existing } }
""")
try:
    plain_not.plan()
except StratificationError as e:
    print("Case 3 - the same rule with a plain NOT cannot be ordered:", type(e).__name__)

Case 1 - base graph has no priority. Derived:
@prefix ex: <http://example.org/> .

ex:db ex:reviewPriority ex:high .

Case 2 - base graph already has ex:db ex:reviewPriority ex:low. Triples derived: 0
Case 3 - the same rule with a plain NOT cannot be ordered: StratificationError


## 9. Importing rule sets

A rule set can import other rule sets with `IMPORTS <iri>`, so rules can be kept in shared libraries. Importing means reading a document, which has security implications, so StarLayer never reads one unless you say how. There are three steps:

1. **Have the libraries as SRL text.** They can come from anywhere: files, a verified copy, a database. Here they are Python strings.
2. **Write a loader.** A loader is a function that takes the IRI of an import and returns that library's text. It is the one place that decides where libraries come from.
3. **Pass the loader to `infer()`** (or `query()` or `plan()`) as `imports_loader=`.

The imports are resolved before any rule runs, each time you call `infer()`. A document is read **once** per call, even if several rule sets import it or the imports form a cycle, and the imported rules and `DATA` are merged into the rule set.

Each rule in the example has an id, written after `RULE`, as in `RULE ex:directExposure { ... } WHERE { ... }`. An id is how a rule is named, for example in a plan. It is optional.

Here a base library defines the exposure rule, a second library builds on it, and our rule set imports the second one:

In [13]:
# Step 1: the libraries, as SRL text
base_library = """
PREFIX ex: <http://example.org/>
RULE ex:directExposure { ?x ex:exposedTo ?v } WHERE { ?x ex:hasVulnerability ?v }
"""

exposure_library = """
PREFIX ex: <http://example.org/>
IMPORTS <http://example.org/lib/base.srl>
RULE ex:inheritedExposure { ?x ex:exposedTo ?v } WHERE { ?x ex:dependsOn ?y . ?y ex:exposedTo ?v }
"""

libraries = {
    "http://example.org/lib/base.srl": base_library,
    "http://example.org/lib/exposure.srl": exposure_library,
}


# Step 2: the loader - given the IRI of an import, returns that library's text
def library_loader(iri):
    return libraries[iri]


# The rule set to run: it imports the exposure library and adds one rule of its own
with_imports = srl.parse_ruleset("""
PREFIX ex: <http://example.org/>
IMPORTS <http://example.org/lib/exposure.srl>
RULE ex:markCritical { ?x ex:status ex:criticallyExposed } WHERE { ?x ex:exposedTo ?v }
""")

# Step 3: pass the loader to infer()
imported = with_imports.infer(base, imports_loader=library_loader)
imported.bind("ex", EX)


show(imported)

assert (ex("frontend"), ex("status"), ex("criticallyExposed")) in imported

@prefix ex: <http://example.org/> .

ex:app ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:db ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .

ex:frontend ex:exposedTo ex:vuln1 ;
    ex:status ex:criticallyExposed .



To see what the imports contributed, `srl.resolve_imports()` returns the rule set with the imported rules merged in and no imports left. Resolving once and running the result many times avoids reading the libraries on every `infer()`:

In [14]:
resolved = srl.resolve_imports(with_imports, library_loader)
print(f"{len(with_imports.rules)} rule written, {len(resolved.rules)} after resolving the imports; imports left: {resolved.imports}")

# the resolved rule set needs no loader
assert len(resolved.infer(base)) == len(imported)
assert len(resolved.rules) == 3

1 rule written, 3 after resolving the imports; imports left: []


`plan()` takes the same `imports_loader=`, so you can see how the rules will run *after* the imports are resolved. Each rule here has an id so the plan can name it. The plan lists the two imported rules, `ex:directExposure` and `ex:inheritedExposure`, together with the rule written in this rule set, `ex:markCritical`, and the edges show which depends on which. All three share one stratum because none of them uses `NOT`; a dependency only forces a later stratum when it is through a `NOT`.

In [15]:
plan = with_imports.plan(imports_loader=library_loader)

names = {r.index: base.qname(r.id) for stratum in plan.strata for r in stratum.general + stratum.run_once}

for stratum in plan.strata:
    print(f"stratum {stratum.number}:", ", ".join(names[r.index] for r in stratum.general))
for edge in sorted(plan.edges, key=lambda e: (e.source, e.target)):
    print(f"{names[edge.source]} depends on {names[edge.target]} ({edge.label})")

assert set(names.values()) == {"ex:markCritical", "ex:directExposure", "ex:inheritedExposure"}

stratum 0:

 ex:markCritical, ex:inheritedExposure, ex:directExposure
ex:markCritical depends on ex:inheritedExposure (open)
ex:markCritical depends on ex:directExposure (open)
ex:inheritedExposure depends on ex:inheritedExposure (open)
ex:inheritedExposure depends on ex:directExposure (open)
